<a href="https://colab.research.google.com/github/mohamedaamar744-ux/flyrank-ml-internship/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedaamar744-ux/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(len(HF_TOKEN))

37


In [ ]:
from huggingface_hub import whoami

print(whoami(token=HF_TOKEN))

{'type': 'user', 'id': '6aaae4b2081b65e8c12a0f15', 'name': 'mohamed-amar', 'fullname': 'mohamed _amar', 'email': 'mohamedaamar744@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/298e65f10045a6752f77fe8454dc0618.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank-colab', 'role': 'read', 'createdAt': '2026-09-16T21:27:14.220Z'}}}


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of analysis:
One row represents one report date × pseudonymized client × pseudonymized content item in fact_content_daily_performance.

Development window:
We use March 2026 as the development window for inspecting the data and building features.

Sealed test window:
The _sample release represents the latest full month, June 2026. We treat June as a sealed test month and do not use it for feature development or decision-making.

The goal of this contract is to define the observation unit and time boundary before feature construction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

con = duckdb.connect()

con.execute("""
CREATE SECRET IF NOT EXISTS (
    TYPE huggingface,
    TOKEN ?
)
""", [HF_TOKEN])

rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬──────────────────┬────────────────────────┐
│ total_rows │ distinct_dates │ distinct_clients │ distinct_content_items │
│   int64    │     int64      │      int64       │         int64          │
├────────────┼────────────────┼──────────────────┼────────────────────────┤
│    9841378 │             31 │               55 │                 331437 │
└────────────┴────────────────┴──────────────────┴────────────────────────┘

In [ ]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
        AS duplicate_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ duplicate_grain_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │                    0 │
└────────────┴──────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Feature

These fields can be used as observed performance signals for content opportunity scoring:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_sum_position`
* `gsc_avg_position`
* `ga4_pageviews`
* `ga4_sessions`
* `ga4_users`
* `ga4_engaged_sessions`
* `ga4_total_engagement_sec`
* `sessions_organic`
* `sessions_direct`
* `sessions_referral`
* `sessions_social`
* `sessions_paid`
* `sessions_ai`
* `ai_chatgpt`
* `ai_perplexity`
* `ai_gemini`
* `ai_copilot`
* `ai_claude`
* `ai_meta`
* `ai_other`
* `scroll_events`

### Label

* No observed label is available in `fact_content_daily_performance`.
* Therefore, this contract does not define a supervised target from this table.

### Context

These fields describe the observation or whether the underlying source data is available:

* `report_date`
* `client_hash_id`
* `content_hash_id`
* `client_has_gsc`
* `client_has_ga4`
* `gsc_data_available`
* `ga4_data_available`

### Excluded

* `month` — excluded because it is the partition field and duplicates the month information already represented by `report_date`.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
))
"""

schema = con.sql(query).df()

print(schema.to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
### Verification results

The March 2026 development window was verified with queries:

* The dataset contains **9,841,378 rows**, covering **31 distinct report dates**, **55 clients**, and **331,437 content items**.
* The observed grain has **0 duplicate rows** for `report_date × client_hash_id × content_hash_id`.
* `report_date`, `client_hash_id`, and `content_hash_id` have **0 missing values** in the March 2026 window.
* `gsc_impressions` and `gsc_clicks` have **0 missing values**.
* `ga4_pageviews` and `ga4_sessions` each have **3,018,741 missing values**, reflecting incomplete GA4 availability.
* The observed date range is **2026-03-01 through 2026-03-31**, with all **31 dates** present.

These checks support the March 2026 development-window and observed-grain claims above.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

┌────────────┬────────────────┬──────────────────┬────────────────────────┐
│ total_rows │ distinct_dates │ distinct_clients │ distinct_content_items │
│   int64    │     int64      │      int64       │         int64          │
├────────────┼────────────────┼──────────────────┼────────────────────────┤
│    9841378 │             31 │               55 │                 331437 │
└────────────┴────────────────┴──────────────────┴────────────────────────┘

In [ ]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
        AS duplicate_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ duplicate_grain_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │                    0 │
└────────────┴──────────────────────┘

In [ ]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(report_date) AS missing_report_date,
    COUNT(*) - COUNT(client_hash_id) AS missing_client_hash_id,
    COUNT(*) - COUNT(content_hash_id) AS missing_content_hash_id,
    COUNT(*) - COUNT(gsc_impressions) AS missing_gsc_impressions,
    COUNT(*) - COUNT(gsc_clicks) AS missing_gsc_clicks,
    COUNT(*) - COUNT(ga4_pageviews) AS missing_ga4_pageviews,
    COUNT(*) - COUNT(ga4_sessions) AS missing_ga4_sessions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────────────┬─────────────────────────┬─────────────────────────┬────────────────────┬───────────────────────┬──────────────────────┐
│ total_rows │ missing_report_date │ missing_client_hash_id │ missing_content_hash_id │ missing_gsc_impressions │ missing_gsc_clicks │ missing_ga4_pageviews │ missing_ga4_sessions │
│   int64    │        int64        │         int64          │          int64          │          int64          │       int64        │         int64         │        int64         │
├────────────┼─────────────────────┼────────────────────────┼─────────────────────────┼─────────────────────────┼────────────────────┼───────────────────────┼──────────────────────┤
│    9841378 │                   0 │                      0 │                       0 │                       0 │                  0 │               3018741 │              3018741 │
└────────────┴─────────────────────┴────────────────────────┴─────────────────────────┴───

In [ ]:
query = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

┌────────────┬────────────┬────────────────┐
│  min_date  │  max_date  │ distinct_dates │
│    date    │    date    │     int64      │
├────────────┼────────────┼────────────────┤
│ 2026-03-01 │ 2026-03-31 │             31 │
└────────────┴────────────┴────────────────┘

In [ ]:
query = f"""
WITH dates AS (
    SELECT DISTINCT report_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
)
SELECT
    COUNT(*) AS observed_dates,
    COUNT(*) - COUNT(DISTINCT report_date) AS duplicate_dates,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM dates
"""

con.sql(query)

┌────────────────┬─────────────────┬────────────┬────────────┐
│ observed_dates │ duplicate_dates │ first_date │ last_date  │
│     int64      │      int64      │    date    │    date    │
├────────────────┼─────────────────┼────────────┼────────────┤
│             31 │               0 │ 2026-03-01 │ 2026-03-31 │
└────────────────┴─────────────────┴────────────┴────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
### Data limits

* **Unbalanced history:** The available history is not necessarily balanced across clients and content items, so observations may not cover the same time span for every client or content item.

* **Source availability:** GA4 metrics are not available for every row. In the March 2026 development window, `ga4_pageviews` and `ga4_sessions` each have 3,018,741 missing values. This means GA4-based signals should be interpreted only where the underlying data is available.

* **GSC-only periods:** In March 2026, **1,718,348 rows** have GSC data available while GA4 data is unavailable, and **1,528,366 rows** have GSC data available while GA4 availability is null. This shows that source availability differs across observations.

* **No supervised outcome:** `fact_content_daily_performance` does not provide an observed target label for content opportunity. Any opportunity score built from these fields should therefore be treated as a scoring or decision-support signal, not as a measured prediction of a known outcome.

* **Time-window limitations:** March 2026 is used for development, while June 2026 is treated as the sealed test month. Features and decisions should not be developed using the sealed month.

* **No causal interpretation:** The observed metrics describe content performance and traffic signals, but they do not by themselves establish that a specific content change caused a change in performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check source-availability combinations in the March 2026 window

query = f"""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    gsc_data_available,
    ga4_data_available
ORDER BY
    gsc_data_available DESC,
    ga4_data_available DESC
"""

con.sql(query)

┌────────────────────┬────────────────────┬─────────┐
│ gsc_data_available │ ga4_data_available │  rows   │
│      boolean       │      boolean       │  int64  │
├────────────────────┼────────────────────┼─────────┤
│ true               │ true               │  364347 │
│ true               │ false              │ 1718348 │
│ true               │ NULL               │ 1528366 │
│ false              │ true               │   49619 │
│ false              │ false              │ 4690323 │
│ false              │ NULL               │ 1490375 │
└────────────────────┴────────────────────┴─────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.